In [1]:
import numpy as np
import torch

In [2]:
# 1. 입력값 X와 정답 y 준비

X = np.array([160, 170, 180, 190])

y = np.array([0, 0, 1, 1])

print('입력값 X:', X)
print('정답값 y:', y)

입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [3]:
# 2. 입력값 정규화

# 평균과 표준편차
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean:', X_mean)
print('입력값 표준편차 X_std:', X_std)
print('정규화된 입력값 X_norm:', X_norm)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.180339887498949
정규화된 입력값 X_norm: [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
# 2-1. X_norm과 y를 PyTorch tensor로 변환하고 shape을 (n, 1)로 정리

# 학습에 사용할 입력값(X_norm)과 정답(y)을 tensor로 바꿔 둠

# 주의(헷갈리기 쉬움):
#   X      = 원래 키(cm)
#   X_norm = 정규화된 입력값  <- 학습에는 이 값을 사용함
# 따라서 아래에서도 X가 아니라 X_norm을 tensor로 변환

# dtype=torch.float32 : 소수점 계산(미분)을 위해 실수(float) 형식으로 만듦
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
# 즉 shape(n, 1)이어야 함. 그래서 reshape(-1, 1)로 모양을 바꿈
#   -1 : 행 개수는 알아서 (여기서는 4)
#    1 : 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print('학습용 입력 tensor X_norm_tensor:\n', X_norm_tensor)
print('학습용 정답 tensor y_tensor:\n', y_tensor)

# shape을 꼭 확인! 둘 다 (4, 1)이어야 함
print('X_norm_tensor shape:', X_norm_tensor.shape)
print('y_tensor shape:', y_tensor.shape)

학습용 입력 tensor X_norm_tensor:
 tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor:
 tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape: torch.Size([4, 1])
y_tensor shape: torch.Size([4, 1])


In [5]:
# 3. torch.nn.Linear(1, 1) 생성 (기존 a, b를 직접 만들지 않음)

# torch.manual_seed(42)
#   - PyTorch의 랜덤 초기값을 고정
#   - nn.Linear는 weight(=a)와 bias(=b)를 랜덤으로 초기화하므로,
#     seed를 고정해야 실행할 때마다 결과가 크게 달라지지 않음
#   - 반드시 linear를 만들기 '직전'에 둠
torch.manual_seed(42)

# torch.nn.Linear(1, 1)
#    - 입력 특성 1개(키 하나)를 받아 출력 1개(H(x) 값 하나)를 내보내는 부품
#    - 내부적으로 H(x) = a * X_norm + b를 계산
#    - 기존 강의의 a는 linear.weight, 기존 강의의 b는 linear.bias
#    - 이전 파일처럼 a, b tensor를 직접 만들지 않는다는 점이 핵심
linear = torch.nn.Linear(1, 1)

# linear 안에 자동으로 만들어진 weight(=a)와 bias(=b)를 확인함
# 학생들이 'a, b가 사라졌다'고 느끼지 않도록, 학습 '전'의 초기값을 직접 봐 둠
print('linear.weight:', linear.weight) # 기존 강의의 a 역할
print('linear.bias:', linear.bias)     # 기존 강의의 b 역할


linear.weight: Parameter containing:
tensor([[0.7645]], requires_grad=True)
linear.bias: Parameter containing:
tensor([0.8300], requires_grad=True)


In [6]:
# 4. torch.nn.BCELoss() 생성 (직접 BCE 수식을 적지 않음)

# torch.nn.BCELoss()
#   - Binary Cross Entropy Cost를 계산해 주는 PyTorch 부품
#   - 사용법: mean_cost = criterion(z, y_tensor)
#   - 주의: 입력으로 H가 아니라, sigmoid를 통과한 예측 확률 z를 넣어야 함
#           (H를 넣는 BCEWithLogitsLoss는 이번 노트북에서 사용하지 않음)
#   - 변수 이름은 관례적으로 criterion(판정 기준)이라고 지음
criterion = torch.nn.BCELoss()

print('criterion 준비 완료:', criterion)

criterion 준비 완료: BCELoss()


In [7]:
# 5. 학습 설정 (learning_rate, epochs)

# learning_rate(학습률): 한 번ㄴ에 weight, bias를 얼마나 크게 수정할지 정하는 값
# 이 값은 바로 다음 셀에서 optimizer를 만들 때 lr=learning_rate로 넘겨줌
learning_rate = 0.1

# epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할지 정하는 값
# 여기서는 같은 데이터로 1000번 반복 학습
epochs = 1000

print('learning_rate:', learning_rate)
print('epochs:', epochs)

learning_rate: 0.1
epochs: 1000


In [8]:
# 6. optimizer 생성 (linear.parameters()를 넘김)

# torch.optim.SGD(linear.parameters(), lr=learning_rate)
#   - linear.parameters(): optimizer가 업데이트할 학습 대상
#                          linear 안의 weight(=a)와 bias(=b)를 가져옴
#   - lr=learning_rate   : 한 번에 얼마나 움직일지(학습률). 위에서 정한 0.1을 넘겨줌

# 이전 파일에서는 [a, b]를 직접 넘겼지만, 이번에는 a, b가 linear 안에 있으므로 
# linear.parameters()를 넘긴다는 점이 핵심
optimizer = torch.optim.SGD(linear.parameters(), lr=learning_rate)

# linear.parameters()가 실제로 무엇을 가져오는지 눈으로 확인
# 이 출력 결과에 보이는 값들이 optimizer가 업데이트할 학습 대상
# (첫 번째가 weight(=a), 두 번째가 bias(=b)
print(list(linear.parameters()))

[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [9]:
# 7. nn.Linear + nn.BCELoss로 경사하강법 학습 (이번 실습의 핵심 루프)

# 한 번의 epoch에서 일어나는 단계 (반드시 이 순서):
#  1. optimizer.zero_grad()      : 이전 epoch의 grad를 0으로 초기화
#  2. H - linear(X_norm_tensor)  : torch.nn.Linear로 H(x) = a*X_norm + b 계산
#  3. z - torch.sigmoid(H)       : 예측 확률
#  4. mean_cost = criterion(z, y): torch.nn.BCELoss로 Cost 계산 (z를 넣음! H 아님)
#  5. mean_cost.backward()       : linear.weight.grad, linear.bias.grad 자동 계산
#  6. optimizer.step()           : linear.weight, linear.bias 업데이트
#  7. 학습 상태 출력

for epoch in range(epochs):
    
    # ---- 1. 이전 epoch에서 계산된 grad 초기화 ----
    # PyTorch의 grad는 덮어쓰기가 아니라 '누적(더하기)'됨
    # optimizer는 linear의 weight, bias를 관리하므로, 
    # 이 한 줄이 linear의.weight.grad, linear의.bias.grad를 한꺼번에 0으로 만듦
    optimizer.zero_grad()
    
    # ---- 2. H(x) = a * X_norm + b 계산 ----
    # linear(X_norm_tensor)를 실행하면 PyTorch가 내부적으로 H(x) = a*X_norm + b를 계산
    # 이때 a는 linear.weight, b는 linear.bias임
    # H(x)는 sigmoid에 들어가기 전의 선형 계산값임 (확률이 아님)
    H = linear(X_norm_tensor)
    
    # ---- 3. sigmoid를 적용해 예측 확률 z 계산 ----
    # z는 0~1 사이의 예측 확률
    z = torch.sigmoid(H)
    
    # ---- 4. torch.nn.BCELoss로 Cost 계산 ----
    # 주의: criterion에는 H가 아니라, sigmoid를 통과한 z를 넣음!
    #       (H를 넣는 것은 나중에 BCEWithLogitsLoss 방식이며, 이번 파일에서는 쓰지 않음)
    mean_cost = criterion(z, y_tensor)
    
    # ---- 5. backward: linear.weight.grad, linear.bias.grad 자동 계산 ----
    # mean_cost에서 출발해 a, b까지 거꾸로 따라가며 미분값을 구해
    #   linear.weight.grad (= 기존 grad_a)
    #   linear.bias.grad   (= 기존 grad_b)
    # 에 저장
    mean_cost.backward()
    
    # ---- 6. optimizer가 weight와 bias 업데이트 ----
    # optimizer.step()은 linear.weight.grad, linear.bias.grad를 사용해
    # Cost가 줄어드는 방향으로 linear.weight, linear.bias를 수정
    optimizer.step()
    
    # ---- 7. 학습 상태 출력
    # 입력 특성이 1개라 weight, bias에 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냄
    # 100 epoch마다 한 번씩, 그리고 마지막 epoch에서 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(
            f'epoch={epoch}, '
            f'Cost={mean_cost.item():.6f}, '
            f'weight(a)={linear.weight.item():.6f}, '
            f'bias(b)={linear.bias.item():.6f}'
        )
    
    # (참고) 초반 3 epoch에서만 a.grad, b.grad 값을 확인해 봄
    # 기존 autograd 실습에서 a.grad, b.grad를 확인하던 것을,
    # 이번에는 linear.weight.grad, linear.bias.grad로 확인
    # 기존 grad_a -> linear.weight.grad
    # 기존 grad_b -> linear.bias.grad
    if epoch < 3:
        print(
            f'  (확인용) linear.weight.grad={linear.weight.grad.item():.6f}, '
            f'linear.bias.grad={linear.bias.grad.item():.6f}'
        )

epoch=0, Cost=0.495464, weight(a)=0.793780, bias(b)=0.812529
  (확인용) linear.weight.grad=-0.292415, linear.bias.grad=0.174793
  (확인용) linear.weight.grad=-0.286153, linear.bias.grad=0.169918
  (확인용) linear.weight.grad=-0.280072, linear.bias.grad=0.165171
epoch=100, Cost=0.178670, weight(a)=2.290082, bias(b)=0.173212
epoch=200, Cost=0.125357, weight(a)=3.002210, bias(b)=0.061586
epoch=300, Cost=0.099283, weight(a)=3.509002, bias(b)=0.026837
epoch=400, Cost=0.082901, weight(a)=3.912263, bias(b)=0.013229
epoch=500, Cost=0.071398, weight(a)=4.250606, bias(b)=0.007116
epoch=600, Cost=0.062789, weight(a)=4.543496, bias(b)=0.004091
epoch=700, Cost=0.056068, weight(a)=4.802371, bias(b)=0.002480
epoch=800, Cost=0.050660, weight(a)=5.034644, bias(b)=0.001570
epoch=900, Cost=0.046207, weight(a)=5.245449, bias(b)=0.001031
epoch=999, Cost=0.042507, weight(a)=5.436657, bias(b)=0.000701


In [10]:
# 8. 학습 완료 후 최종 weight(a), bias(b) 확인

# 학습된 weight, bias는 optimizer.step()에 의해 1000번 반복 업데이트된 값
# (정규화된 입력값 X_norm을 기준으로 학습된 값이라는 점!)

# 입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냄
print('학습된 weight(a):', linear.weight.item())
print('학습된 bias(b):', linear.bias.item())

# tensor 원본 형태도 함께 확인해 둠. (shape과 requires_grad 표시를 볼 수 있음)
print('linear.weight:', linear.weight)
print('linear.bias:', linear.bias)

학습된 weight(a): 5.436656951904297
학습된 bias(b): 0.0007013267604634166
linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)


In [11]:
# 11. 새로운 입력값 예측

# 키가 185cm인 사람이 농구선수인지 예측
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 함
# 학습 때 사용한 X_mean, X_std를 그대로 다시 사용 (새로 계산하면 안 됨)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 a, b를 업데이트하지 않음
# 따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산
with torch.no_grad():
    # torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 함
    #   [[input_norm]] : 이중 대괄호로 감싸 (데이터 1개, 입력 특성 1개) = (1, 1) 형태로 만듦  
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print('input_norm_tensor shape:', input_norm_tensor.shape)
    
    # H(x) = a * X_norm + b (학습된 linear가 계산 - 확률이 아님)
    H_new = linear(input_norm_tensor)
    # z = sigmoid(H) (예측 확률 - 0~1 사이)
    z_new = torch.sigmoid(H_new)
    # 0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    # z_new는 shape (1, 1) tensor이므로 .item()으로 숫자 하나를 꺼냄
    pred = 1 if z_new.item() >= 0.5 else 0

print(f'키가 {input_height}cm인 사람이 농구선수일 확률(z): {z_new.item():.4f}')
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

input_norm_tensor shape: torch.Size([1, 1])
키가 185cm인 사람이 농구선수일 확률(z): 0.9923
판별 결과: 농구선수입니다.
